# 04b · Custodial migration: Coinbase, 2025-11-22

Inputs: tables from 04

Outputs: `custodial_keys`; `key_flags.is_custodial` (merged into `is_service`, original kept as `is_service_base`); `coinbase_migration` in `events`; re-drawn `placebo_draws_strict`; `custodial_rule_check.csv`

Identifies the Coinbase on-chain fund migration of 22–23 November 2025 (*Transparency update: Coinbase executes onchain fund migration*, 2025-11-22; 14:00 UTC on 11-22 to 03:05 UTC on 11-23) and removes the custodial keys involved from the risk set. Coinbase publishes no reserve addresses, so these keys are not covered by the exchange labels of 04.

1. Rule: transactions on 2025-11-22/23 with a single input key and a single output, entirely y1 strict (new address, unspent for 30 days), in which the key spends ≥ `MIG_MIN_BTC` → `custodial_keys`. 05 filters on `NOT is_service`.
2. `coinbase_migration` (2025-11-22, event_set = confound) is added to `events`. Placebo dates whose window covers it are removed (main s ∈ [c−30, c+14], symmetric s ∈ [c−30, c+30], as in 03), the negative control nq_2025-11-21 is dropped, and `placebo_draws_strict` is re-drawn.

Threshold check: rule matches on the two migration days vs the count expected from other tagged days between 2025-10-01 and 2025-12-31 (`custodial_rule_check.csv`):

| Key balance | Matches, 11-22/23 | Expected | Excess share |
|---|---|---|---|
| 1–10 BTC | 51 | about 66 | none |
| 10–100 BTC | 9 | about 11 | none |
| ≥ 100 BTC | 286 | about 14 | 95% |

The threshold is set from this check: below 100 BTC, matches on the migration days do not exceed the ordinary-day expectation (`MIG_MIN_BTC` = 100).

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

RISK_MIN_BTC       = 0.01    # risk set: minimum balance at t-1
RISK_LOOKBACK_DAYS = 365     # risk set: incoming transfer within 12 months (not applied to positive controls)
SERVICE_MAX_RECV   = 1000    # any address with more lifetime receipts -> service/custodian, excluded
MAIN_WIN   = (-14, 30)       # main window
WIDE_WIN   = (-60, 60)       # wider window for main events and positive controls, for pre-trend plots
DORMANT_DAYS = 30            # y1 strict: unspent for the next 30 days
EXPAND_COSPEND   = True      # expand exchange labels by one hop of common-input spending
COSPEND_MAX_IN   = 1000      # expansion uses only transactions with at most this many inputs
N_DRAWS, SEED    = 1000, 20260923
AUDIT_N          = 200       # size of the y1-strict audit sample

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, io, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd
import requests

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["flow_key_tx", "key_flags", "events", "event_flow_coverage", "placebo_days", "placebo_draws_strict"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from 04 found. Spent in this Colab session: $%.2f / $%.0f (the ledger file lives in the current session only)" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def upload(name, df):
    df = df.copy()
    for c in df.columns:                       # dates are always stored as DATE (joinable with the date columns of utxo_life)
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    job = client.load_table_from_dataframe(
        df, f"{DS}.{name}", job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"[{name}] uploaded {len(df):,} rows")

SATS_MIN = int(round(RISK_MIN_BTC * 1e8))
FREEZE_DATE = "2026-09-22"   # freeze date = latest snapshot date (utxo_life has only part of the blocks of 09-23, which are not used)
DATA_END = pd.Timestamp(FREEZE_DATE)
_latest = pd.Timestamp(q(f"SELECT MAX(snap_date) AS d FROM `{DS}.holder_snapshot` WHERE event = 'latest'").d.iloc[0])
assert _latest == DATA_END, f"latest snapshot date {_latest.date()} differs from the freeze date {FREEZE_DATE}"
print("Freeze date (last data day):", DATA_END.date())

## Step 1 · Rule matches: migration days vs ordinary days

In [ ]:
MIG_DATES   = ["2025-11-22", "2025-11-23"]
MIG_EVENT   = "2025-11-22"
MIG_URL     = "https://www.coinbase.com/blog/transparency-update-coinbase-executes-onchain-fund-migration"
BASE_FROM, BASE_TO = "2025-10-01", "2025-12-31"
MIG_MIN_BTC = 100.0   # see custodial_rule_check: no excess below 100 BTC, 95% excess at >= 100 BTC
THRESHOLDS  = [0.01, 0.1, 1, 10, 100]
RULE = "n_in_keys = 1 AND n_out = 1 AND y3 > 0 AND y1_strict >= 0.99 * y3"

cols = ",\n  ".join(f"COUNT(DISTINCT IF(key_in_sats >= {int(t * 1e8)}, key_id, NULL)) AS ge_{str(t).replace('.', '_')}"
                    for t in THRESHOLDS)
hits = q(f'''
SELECT date,
  {cols},
  ROUND(SUM(IF(key_in_sats >= {int(MIG_MIN_BTC * 1e8)}, y1_strict, 0)) / 1e8) AS btc_at_min
FROM `{DS}.flow_key_tx`
WHERE date BETWEEN DATE '{BASE_FROM}' AND DATE '{BASE_TO}' AND {RULE}
GROUP BY date ORDER BY date''')
hits["date"] = pd.to_datetime(hits.date)
is_mig = hits.date.isin(pd.to_datetime(MIG_DATES))
print(f"{len(hits)} tagged days from {BASE_FROM} to {BASE_TO} (only days inside event/placebo windows have data)")
rows = []
for t in THRESHOLDS:
    c = f"ge_{str(t).replace('.', '_')}"
    mig = int(hits.loc[is_mig, c].sum())
    base = float(hits.loc[~is_mig, c].mean())
    rows.append({"min_btc": t, "hits_mig_2days": mig, "base_per_day": round(base, 1),
                 "expected_false_2days": round(2 * base, 1),
                 "excess_share": round(1 - 2 * base / mig, 3) if mig else None})
rule_check = pd.DataFrame(rows)
display(rule_check)
display(hits.loc[is_mig | (hits.date.between("2025-11-15", "2025-11-30"))])
rule_check.to_csv("custodial_rule_check.csv", index=False)

## Step 2 · `custodial_keys`, and update `key_flags`

In [ ]:
SQL_CUST = f'''
CREATE OR REPLACE TABLE `{DS}.custodial_keys` AS
SELECT key_id,
       'coinbase_migration_{MIG_EVENT}' AS src,
       COUNT(*)                         AS n_tx,
       SUM(key_in_sats)                 AS sats,
       MIN(date)                        AS first_date
FROM `{DS}.flow_key_tx`
WHERE date IN ({", ".join(f"DATE '{d}'" for d in MIG_DATES)})
  AND {RULE}
  AND key_in_sats >= {int(MIG_MIN_BTC * 1e8)}
GROUP BY key_id
'''
build("custodial_keys", SQL_CUST, force=True)

names = [f.name for f in client.get_table(f"{DS}.key_flags").schema]
base = "is_service_base" if "is_service_base" in names else "is_service"
drop = sorted({base, "is_service"} | ({"is_custodial"} & set(names)))
SQL_KF = f'''
CREATE OR REPLACE TABLE `{DS}.key_flags` AS
SELECT kf.* EXCEPT({", ".join(drop)}),
       kf.{base}                                  AS is_service_base,
       c.key_id IS NOT NULL                       AS is_custodial,
       (kf.{base} OR c.key_id IS NOT NULL)        AS is_service
FROM `{DS}.key_flags` kf
LEFT JOIN `{DS}.custodial_keys` c USING (key_id)
'''
build("key_flags", SQL_KF, force=True)

cust = q(f'''
SELECT COUNT(*) AS n_custodial_keys, ROUND(SUM(sats)/1e8) AS btc_moved,
       (SELECT COUNTIF(is_custodial) FROM `{DS}.key_flags`)                    AS in_key_flags,
       (SELECT COUNTIF(is_custodial AND in_main) FROM `{DS}.key_flags`)        AS in_main_riskset,
       (SELECT COUNTIF(is_custodial AND NOT is_service_base) FROM `{DS}.key_flags`) AS newly_excluded
FROM `{DS}.custodial_keys`''')
display(cust)
# size of these keys in each main-event snapshot (by group)
display(q(f'''
SELECT h.event, h.grp, COUNT(*) AS n_keys, ROUND(SUM(h.sats)/1e8) AS btc
FROM `{DS}.holder_snapshot` h JOIN `{DS}.custodial_keys` c USING (key_id)
WHERE h.event != 'latest' GROUP BY 1, 2 ORDER BY 1, 2'''))

## Step 3 · Add `coinbase_migration` to the event table, remove affected placebo dates, re-draw

In [ ]:
events = q(f"SELECT * FROM `{DS}.events`")
for c in ["event_date", "alt_date", "snap_date"]:
    events[c] = pd.to_datetime(events[c])
if "coinbase_migration" not in set(events.event):
    new = {c: None for c in events.columns}
    new.update({"event": "coinbase_migration", "event_date": pd.Timestamp(MIG_EVENT), "event_set": "confound",
                "kind": "custody", "date_status": "primary_verified", "date_precision": "day",
                "note": ("Coinbase onchain fund migration (official blog 2025-11-22; 09:00-22:05 EST); "
                         "~800k BTC from existing to new internal wallets; custodial keys flagged in custodial_keys"),
                "in_analysis": False, "confounded": False, "attn_confounded": False,
                "snap_date": pd.Timestamp(MIG_EVENT) - pd.Timedelta(days=1)})
    events = pd.concat([events, pd.DataFrame([new])], ignore_index=True)
    for c in ["in_analysis", "confounded", "attn_confounded"]:
        events[c] = events[c].astype(bool)
    for c in ["event_date", "alt_date", "snap_date"]:
        events[c] = pd.to_datetime(events[c])
    events["block_height"] = pd.to_numeric(events.block_height).astype("Int64")
    upload("events", events)
else:
    print("coinbase_migration is already in events; not added again")

c0 = pd.Timestamp(MIG_EVENT)
cov = q(f"SELECT * FROM `{DS}.event_flow_coverage`")
cov["event_date"] = pd.to_datetime(cov.event_date)
cov["cb_confounded"]       = cov.event_date.between(c0 - pd.Timedelta(days=30), c0 + pd.Timedelta(days=14))
cov["cb_confounded_sym30"] = cov.event_date.between(c0 - pd.Timedelta(days=30), c0 + pd.Timedelta(days=30))
print("Windows covering the Coinbase migration (main definition):", cov.loc[cov.cb_confounded, "event"].tolist())
print("Additionally affected under the symmetric +/-30 definition:", cov.loc[cov.cb_confounded_sym30 & ~cov.cb_confounded, "event"].tolist())
upload("event_flow_coverage", cov)
cov.to_csv("event_flow_coverage.csv", index=False)

pdays = q(f"SELECT kind, date FROM `{DS}.placebo_days`")
pdays["date"] = pd.to_datetime(pdays.date)
ok = cov.set_index("event_date")
n_main = int((events.event_set == "main").sum())
def make_draws(p, name, seed):
    rng = np.random.default_rng(seed)
    out = pd.DataFrame([(name, i + 1, j + 1, d) for i in range(N_DRAWS)
                        for j, d in enumerate(sorted(rng.choice(p, size=n_main, replace=False)))],
                       columns=["pool", "draw_id", "slot", "fake_date"])
    out["fake_date"] = pd.to_datetime(out.fake_date)
    return out
pool_report, draws = {}, []
for i, (pname, kind, flag) in enumerate([("main", "ri_pool", "cb_confounded"), ("sym30", "ri_pool_sym30", "cb_confounded_sym30")]):
    full = sorted(pdays.loc[pdays.kind == kind, "date"])
    keep = [d for d in full if bool(ok.loc[d, "strict_ok"]) and not bool(ok.loc[d, flag])]
    pool_report[pname] = {"before": len(full), "after": len(keep),
                          "dropped": [d.strftime("%Y-%m-%d") for d in full if d not in keep],
                          "by_year": pd.Series([d.year for d in keep]).value_counts().sort_index().to_dict()}
    print(f"pool {pname}: {len(full)} -> {len(keep)}; dropped {pool_report[pname]['dropped']}; by year {pool_report[pname]['by_year']}")
    draws.append(make_draws(keep, pname, SEED + 10 + i))
upload("placebo_draws_strict", pd.concat(draws, ignore_index=True))
nq = pdays[pdays.kind == "nonquantum_news"].date
nq_ok = [d.strftime("%Y-%m-%d") for d in nq if not bool(ok.loc[d, "cb_confounded"])]
print("Negative controls available:", nq_ok, "(nq_2025-11-21 no longer used)")

## Step 4 · Report and download

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "mig_dates": MIG_DATES, "mig_min_btc": MIG_MIN_BTC, "rule": RULE, "source": MIG_URL,
          "rule_check": rule_check.to_dict(orient="records"),
          "custodial": cust.astype(str).to_dict(orient="records"),
          "placebo_pools": pool_report, "negative_controls_ok": nq_ok, "ledger": LEDGER}
with open("build_report_04b.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    for fn in ["build_report_04b.json", "custodial_rule_check.csv", "event_flow_coverage.csv"]:
        files.download(fn)
except Exception:
    print("Report saved to build_report_04b.json")